# 修正鸟瞰坐标后的同刻多行人前向耗时

请用私有 Notebook，选择 GPU，挂载内含 `pinns_sync_latency_source_label_private.bin` 的私有 Dataset。可关闭 Internet。第一段校验三档输入、同刻人群和已训练权重；第二段在当前分配的 GPU 完成 72 格实测并生成 `pinns_sync_latency_source_label_results.zip`。原始同步墙钟与 CUDA event 分布按时刻分别保存。RTX 4090 及 Jetson Orin NX 的目标只适用于所识别的相应实机。

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, tempfile, zipfile

EXPECTED = "ab7308ab016d0cad05bbecc268d0fad1e11ca1cbe2cf066be6e01dea07465ecb"
NAME = "pinns_sync_latency_source_label_private.bin"
def stage_private(mount, stage_dir):
    paths = sorted(mount.rglob(NAME))
    if not paths:
        raise FileNotFoundError(f"Attach the private Dataset containing {NAME}")
    if any(hashlib.sha256(p.read_bytes()).hexdigest() != EXPECTED for p in paths):
        raise ValueError("Private input SHA-256 differs from the frozen bundle")
    with zipfile.ZipFile(paths[0]) as z:
        if z.testzip() is not None:
            raise ValueError("CRC mismatch")
        z.extractall(stage_dir)

working = Path("/kaggle/working")
final = working / "pinns_sync_latency_source_label_results.zip"
if final.exists():
    raise FileExistsError(f"Retain the previous run and start a new session: {final}")
stage_dir = Path(tempfile.mkdtemp(prefix="shortpose-latency-"))
stage_private(Path("/kaggle/input"), stage_dir)
base = [sys.executable, str(stage_dir / "measure_latency.py"), "--repo", str(stage_dir / "model"),
        "--fixture-dir", str(stage_dir / "sync"), "--device", "cuda:0"]
subprocess.run([*base, "--check-only", "--output", str(stage_dir / "checked.json")], check=True)

In [ ]:
subprocess.run([*base, "--backend", "eager", "--output", str(stage_dir / "timing.json")], check=True)
report = json.loads((stage_dir / "timing.json").read_text())
if (report["status"] != "cuda_timing" or len(report["cells"]) != 72
        or any(len(s) != 100 for c in report["cells"] for s in c["raw_ms"].values())):
    raise ValueError("Incomplete timing run")
fd, temporary = tempfile.mkstemp(dir=working, prefix=".pinns_sync_", suffix=".zip")
os.close(fd)
try:
    with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as z:
        for name in ("checked.json", "timing.json"):
            z.write(stage_dir / name, arcname=name)
    with zipfile.ZipFile(temporary) as z:
        if z.testzip() is not None:
            raise ValueError("Output CRC mismatch")
    os.link(temporary, final)
finally:
    os.unlink(temporary)
print("Selected GPU:", report["hardware"]["device_name"])
print("Matched proposal budget:", report["hardware"]["proposal_budget"])
print("Saved", final)